In [ ]:
import pandas as pd
import sqlite3

# 1. Crear la base de datos y cargar las 5 tablas limpias
conn = sqlite3.connect('../data/processed/olist_ecommerce.db')

pd.read_csv('../data/processed/orders_clean.csv').to_sql('orders', conn, if_exists='replace', index=False)
pd.read_csv('../data/processed/items_clean.csv').to_sql('items', conn, if_exists='replace', index=False)
pd.read_csv('../data/processed/customers_clean.csv').to_sql('customers', conn, if_exists='replace', index=False)
pd.read_csv('../data/processed/products_clean.csv').to_sql('products', conn, if_exists='replace', index=False)
pd.read_csv('../data/processed/reviews_clean.csv').to_sql('reviews', conn, if_exists='replace', index=False)

# Consulta 1: ¿Cómo impacta la demora logística en la satisfacción del cliente?
# Cruzar ORDERS + REVIEWS usando LEFT JOIN
q_logistica_reviews = """
SELECT
    CASE WHEN o.is_delayed = 1 THEN 'Con Retraso (Delayed)' ELSE 'A Tiempo (On-Time)' END AS estado_entrega,
    COUNT(DISTINCT o.order_id) AS total_pedidos,
    ROUND(AVG(o.delivery_days), 1) AS promedio_dias_envio,
    ROUND(AVG(r.review_score), 2) AS calificacion_promedio_estrellas
FROM orders o
LEFT JOIN reviews r ON o.order_id = r.order_id
WHERE o.order_status = 'delivered' AND o.delivery_days IS NOT NULL
GROUP BY o.is_delayed;
"""

# Consulta 2: Top 10 Categorías de Productos por Facturación Total y Volumen de Ventas
# Cruzar ORDERS + ITEMS + PRODUCTS mediante doble INNER JOIN
q_top_categorias = """
SELECT
    p.product_category_name AS categoria,
    COUNT(DISTINCT i.order_id) AS cantidad_pedidos,
    COUNT(i.order_item_id) AS unidades_vendidas,
    ROUND(SUM(i.price), 2) AS ingresos_totales_brl,
    ROUND(AVG(i.freight_value), 2) AS costo_envio_promedio
FROM items i
INNER JOIN orders o ON i.order_id = o.order_id
INNER JOIN products p ON i.product_id = p.product_id
WHERE o.order_status = 'delivered'
GROUP BY p.product_category_name
ORDER BY ingresos_totales_brl DESC
LIMIT 10;
"""

# Consulta 3: Top 10 Estados con mayor facturación y su tasa de demoras
# Cruzar ORDERS + ITEMS + CUSTOMERS
q_estados = """
SELECT
    c.customer_state AS estado,
    COUNT(DISTINCT o.order_id) AS total_pedidos,
    ROUND(SUM(i.price), 2) AS ingresos_totales_brl,
    ROUND(100.0 * SUM(o.is_delayed) / COUNT(o.order_id), 2) AS tasa_retraso_pct
    FROM orders o
    INNER JOIN customers c ON o.customer_id = c.customer_id
    INNER JOIN items i ON o.order_id = i.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY c.customer_state
    ORDER BY ingresos_totales_brl DESC
    LIMIT 10;
"""

# Consulta 4: Análisis de categorías críticas
# Cruzar ITEMS + PRODUCTS + REVIEWS + ORDERS
q_peores_categorias = """
SELECT
    p.product_category_name AS categoria,
    COUNT(DISTINCT o.order_id) AS total_pedidos,
    ROUND(AVG(r.review_score), 2) AS calificacion_promedio,
    ROUND(100.0 * SUM(o.is_delayed) / COUNT(DISTINCT o.order_id), 2) AS tasa_retraso_pct
FROM items i
INNER JOIN orders o ON i.order_id = o.order_id
INNER JOIN products p ON i.product_id = p.product_id
INNER JOIN reviews r ON o.order_id = r.order_id
WHERE o.order_status = 'delivered'
GROUP BY p.product_category_name
HAVING COUNT(DISTINCT o.order_id) >= 500
ORDER BY calificacion_promedio ASC
LIMIT 10;
"""

print("--- 1. IMPACTO DE LAS DEMORAS EN LA CALIFICACIÓN ---")
display(pd.read_sql_query(q_logistica_reviews, conn))

print("--- 2. TOP 10 CATEGORÍAS POR INGRESOS ---")
display(pd.read_sql_query(q_top_categorias, conn))

print("--- 3. TOP 10 ESTADOS POR FACTURACIÓN Y DEMORAS")
display(pd.read_sql_query(q_estados, conn))

print("--- 4. CATEGORÍAS CON PEOR SATISFACCIÓN ---")
display(pd.read_sql_query(q_peores_categorias, conn))

--- 1. IMPACTO DE LAS DEMORAS EN LA CALIFICACIÓN ---


,estado_entrega,total_pedidos,promedio_dias_envio,calificacion_promedio_estrellas
0,A Tiempo (On-Time),89936,10.5,4.29
1,Con Retraso (Delayed),6534,33.5,2.27


--- 2. TOP 10 CATEGORÍAS POR INGRESOS ---


,categoria,cantidad_pedidos,unidades_vendidas,ingresos_totales_brl,costo_envio_promedio
0,beleza_saude,8647,9465,1233131.72,18.91
1,relogios_presentes,5495,5859,1166176.98,16.75
2,cama_mesa_banho,9272,10953,1023434.76,18.42
3,esporte_lazer,7530,8431,954852.55,19.38
4,informatica_acessorios,6530,7644,888724.61,18.84
5,moveis_decoracao,6307,8160,711927.69,20.64
6,utilidades_domesticas,5743,6795,615628.69,21.01
7,cool_stuff,3559,3718,610204.10,21.91
8,automotivo,3810,4140,578966.65,21.86
9,brinquedos,3804,4030,471286.48,18.80


--- 3. TOP 10 ESTADOS POR FACTURACIÓN Y DEMORAS


,estado,total_pedidos,ingresos_totales_brl,tasa_retraso_pct
0,SP,40501,5067633.16,4.40
1,RJ,12350,1759651.13,11.62
2,MG,11354,1552481.83,4.37
3,RS,5345,728897.47,5.84
4,PR,4923,666063.51,3.88
5,SC,3546,507012.13,8.18
6,BA,3256,493584.14,11.89
7,DF,2080,296498.41,5.99
8,GO,1957,282836.70,6.24
9,ES,1995,268643.45,10.74


--- 4. CATEGORÍAS CON PEOR SATISFACCIÓN ---


,categoria,total_pedidos,calificacion_promedio,tasa_retraso_pct
0,moveis_escritorio,1244,3.51,10.53
1,cama_mesa_banho,9177,3.92,8.13
2,unknown,1382,3.94,8.25
3,moveis_decoracao,6260,3.95,8.93
4,informatica_acessorios,6499,3.99,7.45
5,telefonia,4069,3.99,7.47
6,eletronicos,2498,4.07,8.21
7,relogios_presentes,5454,4.07,7.54
8,bebes,2787,4.08,8.00
9,ferramentas_jardim,3428,4.08,7.99
